In [ ]:
# EX 1 Creation du topic : 

In [7]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers='kafka:9092')

admin.create_topics([NewTopic(name='weather_stream', num_partitions=1, replication_factor=1)])
print("Topic crée")

print(admin.list_topics())
admin.close()


Topic crée
['weather_stream', '__consumer_offsets']


In [ ]:
# EX 1 Envoi du message :

In [8]:
import json
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='kafka:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

producer.send('weather_stream', {"msg": "Hello Kafka"})
producer.flush()  
print("Message envoyé")


Message envoyé


In [5]:
# EX 2 : Creation du consommateur avec variable pour le topic + réception du message: 

In [10]:
from kafka import KafkaConsumer
topic = 'weather_stream'
consumer = KafkaConsumer(
    topic,
    bootstrap_servers='kafka:9092',
    auto_offset_reset='earliest',   
    consumer_timeout_ms=5000        
)
for message in consumer:
    print(message.value.decode())

print("Fin de la lecture.")


{"msg": "Hello Kafka"}
{"msg": "Hello Kafka"}
{"latitude": 48.853382, "longitude": 2.355208, "generationtime_ms": 0.07045269012451172, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 36.0, "current_units": {"time": "iso8601", "interval": "seconds", "temperature_2m": "\u00b0C", "wind_speed_10m": "m/s", "weather_code": "wmo code"}, "current": {"time": "2026-11-10T13:45", "interval": 900, "temperature_2m": 12.3, "wind_speed_10m": 1.91, "weather_code": 80}, "city": "Paris", "country": "France"}
{"latitude": 37.364082, "longitude": -5.956622, "generationtime_ms": 0.053882598876953125, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 18.0, "current_units": {"time": "iso8601", "interval": "seconds", "temperature_2m": "\u00b0C", "wind_speed_10m": "m/s", "weather_code": "wmo code"}, "current": {"time": "2026-11-10T13:45", "interval": 900, "temperature_2m": 21.4, "wind_speed_10m": 5.8, "weather_code": 80}, "city": "

In [ ]:
# EX 3 : Producteur current weather qui interroge l'api toutes les 30 secondes

In [18]:
import json
import time
import requests
from kafka import KafkaProducer

CITIES = [
    {"city": "Paris", "country": "France","latitude": 48.8566, "longitude": 2.3522},
    {"city": "Seville", "country": "Spain", "latitude": 37.3891, "longitude": -5.9845},
    {"city": "Dubai", "country": "United Arab Emirates", "latitude": 25.2048,  "longitude": 55.2708},
    {"city": "Reykjavik", "country": "Iceland",    "latitude": 64.1466, "longitude": -21.9426},
    {"city": "Wellington","country": "New Zealand","latitude": -41.2865, "longitude": 174.7762},
]

URL = ("https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m,weather_code&wind_speed_unit=ms")

while True:
    for c in CITIES:
        try:
            url = URL.format(latitude=c["latitude"], longitude=c["longitude"])
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            data = response.json()
            data["city"] = c["city"]
            data["country"] = c["country"]
            producer.send('weather_stream', data)
            print(f"Envoyé : {c['city']} → {data['current']['temperature_2m']} °C")
        except Exception as e:
            print(f"Erreur pour {c['city']} : {e}")

    producer.flush()
    time.sleep(30)


Envoyé : Paris → 12.5 °C
Envoyé : Seville → 21.0 °C
Envoyé : Dubai → 30.1 °C
Envoyé : Reykjavik → 6.7 °C
Envoyé : Wellington → 15.5 °C
Envoyé : Paris → 12.5 °C
Envoyé : Seville → 21.0 °C
Envoyé : Dubai → 30.1 °C
Envoyé : Reykjavik → 6.7 °C
Envoyé : Wellington → 15.5 °C


KeyboardInterrupt: 

In [ ]:
# 4 transformation des données et detections d'alertes :

In [19]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("weather_stream")
         .master("spark://spark-master:7077")
         .getOrCreate())

kafka_stream = (spark.readStream
       .format("kafka")
       .option("kafka.bootstrap.servers", "kafka:9092")
       .option("subscribe", "weather_stream")
       .option("startingOffsets", "earliest") 
       .load())


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col, from_json

schema = StructType([
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("current", StructType([
        StructField("time", StringType()),
        StructField("temperature_2m", DoubleType()),
        StructField("wind_speed_10m", DoubleType()),
        StructField("weather_code", IntegerType()),
    ])),
])

weather = (kafka_stream
    .select(
        from_json(col("value").cast("string"), schema).alias("data"),
        col("timestamp").alias("event_time")          # heure d'écriture du message dans Kafka
    )
    .select(
        col("event_time"),
        col("data.city").alias("city"),
        col("data.country").alias("country"),
        col("data.current.temperature_2m").alias("temperature"),
        col("data.current.wind_speed_10m").alias("windspeed"),
        col("data.current.weather_code").alias("weather_code"),
    )
    .filter(col("city").isNotNull()))

query = (weather.writeStream
         .format("memory")
         .queryName("weather")
         .outputMode("append")
         .start())

In [29]:
spark.sql("SELECT * FROM weather").show(truncate=False)


+-----------------------+----------+--------------------+-----------+---------+------------+
|event_time             |city      |country             |temperature|windspeed|weather_code|
+-----------------------+----------+--------------------+-----------+---------+------------+
|2026-11-10 13:52:40.233|Paris     |France              |12.3       |1.91     |80          |
|2026-11-10 13:52:40.383|Seville   |Spain               |21.4       |5.8      |80          |
|2026-11-10 13:52:40.514|Dubai     |United Arab Emirates|30.5       |3.27     |0           |
|2026-11-10 13:52:40.713|Reykjavik |Iceland             |6.8        |6.8      |3           |
|2026-11-10 13:52:40.882|Wellington|New Zealand         |15.4       |8.04     |51          |
|2026-11-10 13:53:11.01 |Paris     |France              |12.3       |1.91     |80          |
|2026-11-10 13:53:11.126|Seville   |Spain               |21.4       |5.8      |80          |
|2026-11-10 13:53:11.243|Dubai     |United Arab Emirates|30.5       |3

In [ ]:
# Création du topic weather transformed

In [30]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers='kafka:9092')
try:
    admin.create_topics([NewTopic(name='weather_transformed', num_partitions=1, replication_factor=1)])
    print("Topic créé")
except TopicAlreadyExistsError:
    print("Le topic existe déjà")
print(admin.list_topics())
admin.close()


Le topic existe déjà
['weather_stream', 'weather_transformed', '__consumer_offsets']


In [31]:
from pyspark.sql.functions import when

weather_alerts = (weather
    .withColumn("wind_alert_level",
        when(col("windspeed") < 10, "level_0")
        .when(col("windspeed") <= 20, "level_1")
        .otherwise("level_2"))
    .withColumn("heat_alert_level",
        when(col("temperature") < 25, "level_0")
        .when(col("temperature") <= 35, "level_1")
        .otherwise("level_2")))

In [32]:
from pyspark.sql.functions import to_json, struct

to_kafka = weather_alerts.select(
    to_json(struct("*")).alias("value")
)


In [34]:
query_kafka = (to_kafka.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "weather_transformed")
    .option("checkpointLocation", "/home/jovyan/work/checkpoints/weather_transformed")
    .start())


In [35]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'weather_transformed',
    bootstrap_servers='kafka:9092',
    auto_offset_reset='earliest',
    consumer_timeout_ms=10000
)
for message in consumer:
    print(message.value.decode())


{"city":"Paris","country":"France","time":"2026-11-10T13:45","temperature":12.3,"wind_speed":1.91,"weather_code":80,"wind_alert":"level_0","heat_alert":"level_0"}
{"city":"Seville","country":"Spain","time":"2026-11-10T13:45","temperature":21.4,"wind_speed":5.8,"weather_code":80,"wind_alert":"level_0","heat_alert":"level_0"}
{"city":"Dubai","country":"United Arab Emirates","time":"2026-11-10T13:45","temperature":30.5,"wind_speed":3.27,"weather_code":0,"wind_alert":"level_0","heat_alert":"level_1"}
{"city":"Reykjavik","country":"Iceland","time":"2026-11-10T13:45","temperature":6.8,"wind_speed":6.8,"weather_code":3,"wind_alert":"level_0","heat_alert":"level_0"}
{"city":"Wellington","country":"New Zealand","time":"2026-11-10T13:45","temperature":15.4,"wind_speed":8.04,"weather_code":51,"wind_alert":"level_0","heat_alert":"level_0"}
{"city":"Paris","country":"France","time":"2026-11-10T13:45","temperature":12.3,"wind_speed":1.91,"weather_code":80,"wind_alert":"level_0","heat_alert":"level_0

In [ ]:
# 5 Agrégat en temps réel avec spark:

In [36]:
transformed_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather_transformed")
    .option("startingOffsets", "earliest")
    .load())


In [38]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.functions import col, from_json

schema_transformed = StructType([
    StructField("event_time", TimestampType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("temperature", DoubleType()),
    StructField("windspeed", DoubleType()),
    StructField("weather_code", IntegerType()),
    StructField("wind_alert_level", StringType()),
    StructField("heat_alert_level", StringType()),
])

weather_transformed = (transformed_stream
    .select(from_json(col("value").cast("string"), schema_transformed).alias("data"))
    .select("data.*")
    .filter(col("wind_alert_level").isNotNull()))


In [46]:
query_t = (weather_transformed.writeStream
    .format("memory")
    .queryName("weather_transformed")
    .outputMode("append")
    .start())


In [47]:
spark.sql("SELECT * FROM weather_transformed").show(truncate=False)


+-----------------------+----------+--------------------+-----------+---------+------------+----------------+----------------+
|event_time             |city      |country             |temperature|windspeed|weather_code|wind_alert_level|heat_alert_level|
+-----------------------+----------+--------------------+-----------+---------+------------+----------------+----------------+
|2026-11-10 13:52:40.233|Paris     |France              |12.3       |1.91     |80          |level_0         |level_0         |
|2026-11-10 13:52:40.383|Seville   |Spain               |21.4       |5.8      |80          |level_0         |level_0         |
|2026-11-10 13:52:40.514|Dubai     |United Arab Emirates|30.5       |3.27     |0           |level_0         |level_1         |
|2026-11-10 13:52:40.713|Reykjavik |Iceland             |6.8        |6.8      |3           |level_0         |level_0         |
|2026-11-10 13:52:40.882|Wellington|New Zealand         |15.4       |8.04     |51          |level_0         |le

In [ ]:
# sliding window de 1 min qui avance toute les 30 sec 

In [ ]:
from pyspark.sql.functions import window, avg, count, max as spark_max

weather_windowed = (weather_transformed
    .withWatermark("event_time", "1 minute")
    .groupBy(
        window(col("event_time"), "1 minute", "30 seconds"),
        col("city")
    )
    .agg(
        count("*").alias("nb_messages"),
        avg("temperature").alias("avg_temperature"),
        avg("windspeed").alias("avg_windspeed"),
    ))


In [42]:
weather_windowed_flat = weather_windowed.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    "city", "nb_messages", "avg_temperature", "avg_windspeed"
)


In [43]:
query_win = (weather_windowed_flat.writeStream
    .format("memory")
    .queryName("weather_win")
    .outputMode("append")
    .start())


In [45]:
spark.sql("SELECT * FROM weather_win ORDER BY window_start, city").show(truncate=False)

+-------------------+-------------------+----------+-----------+---------------+-------------+
|window_start       |window_end         |city      |nb_messages|avg_temperature|avg_windspeed|
+-------------------+-------------------+----------+-----------+---------------+-------------+
|2026-11-10 13:52:00|2026-11-10 13:53:00|Dubai     |1          |30.5           |3.27         |
|2026-11-10 13:52:00|2026-11-10 13:53:00|Paris     |1          |12.3           |1.91         |
|2026-11-10 13:52:00|2026-11-10 13:53:00|Reykjavik |1          |6.8            |6.8          |
|2026-11-10 13:52:00|2026-11-10 13:53:00|Seville   |1          |21.4           |5.8          |
|2026-11-10 13:52:00|2026-11-10 13:53:00|Wellington|1          |15.4           |8.04         |
|2026-11-10 13:52:30|2026-11-10 13:53:30|Dubai     |2          |30.5           |3.27         |
|2026-11-10 13:52:30|2026-11-10 13:53:30|Paris     |2          |12.3           |1.91         |
|2026-11-10 13:52:30|2026-11-10 13:53:30|Reykjavik